# DiffusionSat sampler · synthetic minority-class patches

This notebook is the in-notebook twin of `diffusionsat_synth.py`. It
loads `samar-khanna/DiffusionSat` and writes patches to
`data/_cache/synth/<class>/` in the same `.npy + .png` layout the main
`pipeline.ipynb` §6.1 viz cell consumes — so once you've run this
notebook, switch back to the `synthcrop` kernel and re-run §6.1 to
render the new patches.

**Run in the `synthcrop-dsat` env / kernel**, not the main one. DSAT
pins `diffusers==0.18.2` and `huggingface_hub==0.16.4`, which conflict
with the main `synthcrop` env.

One-time setup:

```bash
conda env create -f notebooks/environment-diffusionsat.yml
conda activate synthcrop-dsat
python -m ipykernel install --user --name synthcrop-dsat --display-name "Python (synthcrop-dsat)"
git clone https://github.com/samar-khanna/DiffusionSat.git notebooks/external/DiffusionSat
```

Then in JupyterLab: **Kernel → Change Kernel → Python (synthcrop-dsat)**.

## Imports + paths

In [ ]:
from __future__ import annotations
import importlib, json, sys
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "package.json").exists() and (p / "notebooks").exists():
            return p
    return here


REPO_ROOT = _repo_root()
DSAT_REPO = REPO_ROOT / "notebooks" / "external" / "DiffusionSat"
SYNTH_OUT = REPO_ROOT / "data" / "_cache" / "synth"
SYNTH_OUT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"torch {torch.__version__} on {DEVICE} (dtype {DTYPE})")
print(f"repo  {REPO_ROOT}")
print(f"DSAT  {DSAT_REPO} (exists={DSAT_REPO.exists()})")
print(f"out   {SYNTH_OUT}")


## Compatibility shims

DSAT was authored against `diffusers<=0.18` + `huggingface_hub<=0.16`.
The pinned env tracks those exactly, but a few symbols moved across
patch releases. These shims register both old names so the DSAT import
doesn't fail when the env drifts even slightly.

In [ ]:
def patch_legacy_imports() -> None:
    import sys, types, importlib

    # hf_hub: `cached_download` was removed in huggingface_hub>=0.26.
    try:
        import huggingface_hub as hf
        if not hasattr(hf, "cached_download") and hasattr(hf, "hf_hub_download"):
            hf.cached_download = hf.hf_hub_download
    except Exception as e:
        print(f"note: huggingface_hub shim skipped ({e})")

    # diffusers: DSAT imports `from diffusers.models.cross_attention import
    # AttnProcessor`. In diffusers>=0.18 that module is gone — its symbols
    # were split:
    #   * `AttnProcessor` / `CrossAttnProcessor` -> diffusers.models.attention_processor
    #   * `CrossAttention`                        -> diffusers.models.attention (now `Attention`)
    # Build a synthetic `cross_attention` module that re-exports the union
    # so legacy code stops crashing on import.
    try:
        if "diffusers.models.cross_attention" not in sys.modules:
            attn       = importlib.import_module("diffusers.models.attention")
            attn_proc  = importlib.import_module("diffusers.models.attention_processor")
            shim = types.ModuleType("diffusers.models.cross_attention")
            for src in (attn, attn_proc):
                for name in dir(src):
                    if name.startswith("_"):
                        continue
                    if not hasattr(shim, name):
                        setattr(shim, name, getattr(src, name))
            for old, new in [("CrossAttention", "Attention"),
                             ("CrossAttnProcessor", "AttnProcessor")]:
                if not hasattr(shim, old):
                    for src in (attn_proc, attn):
                        if hasattr(src, new):
                            setattr(shim, old, getattr(src, new))
                            break
            sys.modules["diffusers.models.cross_attention"] = shim
    except Exception as e:
        print(f"note: diffusers shim skipped ({e})")


patch_legacy_imports()

# Sanity check: confirm the legacy import path works before importing DSAT.
import importlib
_legacy = importlib.import_module("diffusers.models.cross_attention")
assert hasattr(_legacy, "AttnProcessor"), "shim missing AttnProcessor"
assert hasattr(_legacy, "CrossAttnProcessor"), "shim missing CrossAttnProcessor"
print("shims applied · cross_attention.AttnProcessor present")


## Sampling config

Adjust the cell below before running. Defaults match the main pipeline:
the same 4 minority class names, 200 patches per class, 40 sampling
steps, classifier-free guidance 7.5.

If you only want a quick smoke run, drop `n` to 8.

In [ ]:
# Class names must match CFG.minority_classes in pipeline.ipynb so
# §6.1 of the main notebook picks up these patches when it reloads.
CLASSES = ("Durian", "Langsat", "Rambutan", "Mangosteen")

N_PER_CLASS = 200       # 8 for smoke runs
STEPS       = 40
GUIDANCE    = 7.5
SEED        = 42

# Centroid of the AOI bbox — used for the model's lat/lng metadata
# embedding. Matches CFG.aoi_bbox SE quadrant from pipeline.ipynb.
AOI_BBOX = (101.55, 12.70, 101.65, 12.80)   # W, S, E, N
DATE_STR = "2024-07-01"

W, S, E, N = AOI_BBOX
LAT, LNG = (S + N) / 2, (W + E) / 2
print(f"AOI center {LAT:.3f}, {LNG:.3f}")
print(f"classes    {CLASSES}")
print(f"per class  {N_PER_CLASS} patches · steps={STEPS} · guidance={GUIDANCE}")


## Load DiffusionSat

First run downloads ~5 GB of weights from HuggingFace and caches them
under `~/.cache/huggingface`. Re-runs skip the download.

In [ ]:
if not DSAT_REPO.exists():
    raise SystemExit(
        f"DiffusionSat repo not found at {DSAT_REPO}.\n"
        f"Clone it:  git clone https://github.com/samar-khanna/DiffusionSat.git {DSAT_REPO}"
    )

# Put the DSAT repo on sys.path before importing.
if str(DSAT_REPO) not in sys.path:
    sys.path.insert(0, str(DSAT_REPO))

from diffusionsat import DiffusionSatPipeline   # noqa: E402

pipe = DiffusionSatPipeline.from_pretrained(
    "samar-khanna/DiffusionSat", torch_dtype=DTYPE,
).to(DEVICE)
print(f"DiffusionSat loaded on {DEVICE}")


## Save helpers

`_pil_to_4band` reconstructs a 4-channel reflectance patch (B02 / B03 /
B04 / B08) from DSAT's RGB output. The NIR channel is synthesised from
a luminance blend — pragmatic placeholder so the main pipeline's
feature extractor keeps working. The synthetic NIR is not physically
meaningful; if you want a real NIR you'll have to either fine-tune
DSAT on 4-band Sentinel-2 patches or fall back to the OpenSR sampler
in the main notebook.

In [ ]:
def _safe_dir(name: str) -> str:
    return name.replace(" ", "_").lower()


def _pil_to_4band(img: Image.Image) -> np.ndarray:
    # arr: (H, W, 3) float in [0, 1].  Output order matches the main
    # notebook: (4, H, W) = (B02, B03, B04, B08).
    arr = np.asarray(img).astype("float32") / 255.0
    R, G, B = arr[..., 0], arr[..., 1], arr[..., 2]
    fake_nir = np.clip(0.5 * G + 0.4 * R + 0.1 * B, 0.0, 1.0)
    return np.stack([B, G, R, fake_nir], axis=0)


def save_patch(class_name: str, idx: int, patch: np.ndarray) -> Path:
    out_dir = SYNTH_OUT / _safe_dir(class_name)
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / f"patch_{idx:03d}.npy", patch.astype("float32"))
    rgb = np.stack([patch[2], patch[1], patch[0]], axis=-1)
    rgb = np.clip(rgb / max(1e-6, np.percentile(rgb, 99)), 0.0, 1.0)
    Image.fromarray((rgb * 255).astype(np.uint8)).save(out_dir / f"patch_{idx:03d}.png")
    return out_dir


## Sample loop

Per-class loop · one prompt + the same lat/lng/ts metadata per patch.
Variation comes from the diffusion sampler's noise schedule. Use a
fresh seed per class so patches don't collide across classes.

In [ ]:
try:
    datetime.fromisoformat(DATE_STR)
except ValueError as e:
    raise SystemExit(f"DATE_STR must be YYYY-MM-DD: {e}")

written: dict[str, Path] = {}
viz_cache: dict[str, list[np.ndarray]] = {}
VIZ_PER_CLASS = 4

for ci, cls in enumerate(CLASSES):
    prompt = f"Sentinel-2 satellite imagery of {cls}, rural Thailand"
    generator = torch.Generator(device=pipe.device).manual_seed(SEED + ci)
    print(f"[{cls}] sampling {N_PER_CLASS} patches")
    viz_cache[cls] = []
    for i in tqdm(range(N_PER_CLASS), desc=cls, leave=False):
        result = pipe(
            prompt=[prompt],
            metadata={"lat": [LAT], "lng": [LNG], "ts": [DATE_STR]},
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE,
            generator=generator,
        )
        patch = _pil_to_4band(result.images[0])
        out_dir = save_patch(cls, i, patch)
        if i < VIZ_PER_CLASS:
            viz_cache[cls].append(patch)
    written[cls] = out_dir
    print(f"[{cls}] -> {out_dir}")

print(f"\nDone. Wrote {sum(N_PER_CLASS for _ in CLASSES)} patches across {len(CLASSES)} classes.")


## Preview

Same RGB / false-colour NIR / NDVI layout the main pipeline's §6.1 uses,
so what you see here is what `pipeline.ipynb` will render after a
kernel switch back to `synthcrop`.

In [ ]:
def _stretch(img: np.ndarray, p: float = 99.0) -> np.ndarray:
    hi = max(1e-6, np.percentile(img, p))
    return np.clip(img / hi, 0.0, 1.0)


def show_grid(cls: str, patches: list[np.ndarray]) -> None:
    if not patches:
        print(f"[{cls}] no patches")
        return
    n = len(patches)
    fig, axes = plt.subplots(n, 3, figsize=(8.5, 2.7 * n), squeeze=False)
    fig.suptitle(f"DSAT synthetic patches · class {cls}", y=1.0, fontsize=11, fontweight="bold")
    eps = 1e-6
    for r, patch in enumerate(patches):
        B02, B03, B04, B08 = patch
        rgb  = _stretch(np.stack([B04, B03, B02], axis=-1))
        nir  = _stretch(np.stack([B08, B04, B03], axis=-1))
        ndvi = (B08 - B04) / (B08 + B04 + eps)
        axes[r, 0].imshow(rgb);  axes[r, 0].set_title("RGB" if r == 0 else "", fontsize=9)
        axes[r, 1].imshow(nir);  axes[r, 1].set_title("false-NIR (synth)" if r == 0 else "", fontsize=9)
        im = axes[r, 2].imshow(ndvi, vmin=-0.2, vmax=0.9, cmap="RdYlGn")
        axes[r, 2].set_title("NDVI" if r == 0 else "", fontsize=9)
        plt.colorbar(im, ax=axes[r, 2], fraction=0.046, pad=0.04)
        for a in axes[r]:
            a.set_xticks([]); a.set_yticks([])
    plt.tight_layout()
    plt.show()


for cls in CLASSES:
    show_grid(cls, viz_cache.get(cls, []))


## Next steps

1. Switch the kernel back to **Python (synthcrop)** in `pipeline.ipynb`.
2. Re-run §6.1 there — the loader will pick up the new patches from
   `data/_cache/synth/<class>/`.
3. Re-run §7 to splice the DSAT-derived feature rows into `DF`. They
   carry the same `_synth = True` flag the OpenSR-derived rows do,
   so §8 RF + §8.1 cascade pick them up without further change.

Tip — if you want to compare base-SR and DSAT patches side by side,
copy the OpenSR outputs to a sibling folder before running this
notebook, e.g. `mv data/_cache/synth data/_cache/synth_opensr` — then
this notebook recreates `data/_cache/synth/` from scratch with the
DSAT outputs.